In [1]:
import numpy as np
from scipy.interpolate import CubicSpline
from datetime import date, timedelta

In [ ]:
def price_storage_contract(
    injection_dates:    list[date],
    withdrawal_dates:   list[date],
    injection_prices:   list[float],
    withdrawal_prices:  list[float],
    injection_rate:     float,        # MMBtu per day (max injectable per date)
    withdrawal_rate:    float,        # MMBtu per day (max withdrawable per date)
    max_volume:         float,        # MMBtu  (storage capacity)
    storage_cost_rate:  float,        # $/MMBtu/day
    verbose:            bool = True,
) -> float:
    # ── 2a. Input validation ──────────────────────────────────────
    if len(injection_dates) != len(injection_prices):
        raise ValueError("injection_dates and injection_prices must have the same length.")
    if len(withdrawal_dates) != len(withdrawal_prices):
        raise ValueError("withdrawal_dates and withdrawal_prices must have the same length.")

    # ── 2b. Build a unified, sorted timeline of all events ───────
    events = []
    for d, p in zip(injection_dates, injection_prices):
        events.append({"date": d, "type": "inject", "price": p})
    for d, p in zip(withdrawal_dates, withdrawal_prices):
        events.append({"date": d, "type": "withdraw", "price": p})

    events.sort(key=lambda e: e["date"])

    # ── 2c. Simulate volume and accumulate cash flows ─────────────
    current_volume  = 0.0   # MMBtu currently in storage
    prev_date       = events[0]["date"] if events else None

    total_injection_cost   = 0.0
    total_withdrawal_revenue = 0.0
    total_storage_cost     = 0.0

    if verbose:
        print("=" * 65)
        print(f"{'Date':<14} {'Event':<10} {'Volume Δ':>10} "
              f"{'Stock':>10} {'Cash Flow':>12}")
        print("-" * 65)

    for ev in events:
        d     = ev["date"]
        etype = ev["type"]
        price = ev["price"]

        # ── Storage cost for the period [prev_date, d) ───────────
        days_elapsed = (d - prev_date).days
        period_storage_cost = current_volume * storage_cost_rate * days_elapsed
        total_storage_cost += period_storage_cost

        # ── Physical transaction ──────────────────────────────────
        if etype == "inject":
            volume = min(injection_rate, max_volume - current_volume)
            if volume <= 0:
                if verbose:
                    print(f"{str(d):<14} {'INJECT':<10} {'SKIPPED (full)':>22}")
                prev_date = d
                continue
            cash_flow = -price * volume          # we BUY gas → negative
            total_injection_cost += abs(cash_flow)
            current_volume += volume

        else:  # withdraw
            volume = min(withdrawal_rate, current_volume)
            if volume <= 0:
                if verbose:
                    print(f"{str(d):<14} {'WITHDRAW':<10} {'SKIPPED (empty)':>22}")
                prev_date = d
                continue
            cash_flow = price * volume           # we SELL gas → positive
            total_withdrawal_revenue += cash_flow
            current_volume -= volume

        if verbose:
            direction = f"+{volume:,.0f}" if etype == "inject" else f"-{volume:,.0f}"
            cf_str = f"${cash_flow:+,.2f}"
            print(f"{str(d):<14} {etype.upper():<10} {direction:>10} "
                  f"{current_volume:>10,.0f} {cf_str:>12}")

        prev_date = d

    # ── 2d. Contract value ────────────────────────────────────────
    contract_value = total_withdrawal_revenue - total_injection_cost - total_storage_cost

    if verbose:
        print("=" * 65)
        print(f"\n  Withdrawal revenue  : ${total_withdrawal_revenue:>12,.2f}")
        print(f"  Injection cost      : ${total_injection_cost:>12,.2f}")
        print(f"  Storage cost        : ${total_storage_cost:>12,.2f}")
        print(f"  ──────────────────────────────────────")
        print(f"  CONTRACT VALUE      : ${contract_value:>12,.2f}")
        print()

    return contract_value



In [4]:
if __name__ == "__main__":

    # ── Build a simple price estimator (from previous task) ───────
    hist_dates  = [date(2024, 1, 1), date(2024, 4, 1),
                   date(2024, 7, 1), date(2024, 10, 1), date(2025, 1, 1)]
    hist_prices = [10.5, 9.8, 8.2, 10.1, 11.3]   # $/MMBtu
    estimate_price = build_price_estimator(hist_dates, hist_prices)


    # ══════════════════════════════════════════
    # TEST 1 — Simple buy-low / sell-high trade
    # ══════════════════════════════════════════
    print("\n" + "─" * 65)
    print("TEST 1 — Basic seasonal trade (inject in summer, withdraw in winter)")
    print("─" * 65)

    t1_inj_dates  = [date(2024, 6, 1), date(2024, 7, 1)]
    t1_with_dates = [date(2024, 11, 1), date(2024, 12, 1)]

    t1_inj_prices  = [estimate_price(d) for d in t1_inj_dates]
    t1_with_prices = [estimate_price(d) for d in t1_with_dates]

    v1 = price_storage_contract(
        injection_dates   = t1_inj_dates,
        withdrawal_dates  = t1_with_dates,
        injection_prices  = t1_inj_prices,
        withdrawal_prices = t1_with_prices,
        injection_rate    = 50_000,     # MMBtu per event
        withdrawal_rate   = 50_000,
        max_volume        = 100_000,    # MMBtu capacity
        storage_cost_rate = 0.0005,     # $/MMBtu/day
    )


    # ══════════════════════════════════════════
    # TEST 2 — Unprofitable contract
    #          (buy high in winter, sell low in summer)
    # ══════════════════════════════════════════
    print("─" * 65)
    print("TEST 2 — Unprofitable trade (inject in winter, withdraw in summer)")
    print("─" * 65)

    t2_inj_dates  = [date(2024, 1, 15)]
    t2_with_dates = [date(2024, 8, 15)]

    t2_inj_prices  = [estimate_price(d) for d in t2_inj_dates]
    t2_with_prices = [estimate_price(d) for d in t2_with_dates]

    v2 = price_storage_contract(
        injection_dates   = t2_inj_dates,
        withdrawal_dates  = t2_with_dates,
        injection_prices  = t2_inj_prices,
        withdrawal_prices = t2_with_prices,
        injection_rate    = 80_000,
        withdrawal_rate   = 80_000,
        max_volume        = 100_000,
        storage_cost_rate = 0.0005,
    )


    # ══════════════════════════════════════════
    # TEST 3 — Multiple injection & withdrawal dates
    # ══════════════════════════════════════════
    print("─" * 65)
    print("TEST 3 — Multi-cycle: 3 injections, 3 withdrawals")
    print("─" * 65)

    t3_inj_dates  = [date(2024, 3, 1), date(2024, 5, 1), date(2024, 6, 15)]
    t3_with_dates = [date(2024, 9, 1), date(2024, 11, 1), date(2025, 1, 1)]

    t3_inj_prices  = [estimate_price(d) for d in t3_inj_dates]
    t3_with_prices = [estimate_price(d) for d in t3_with_dates]

    v3 = price_storage_contract(
        injection_dates   = t3_inj_dates,
        withdrawal_dates  = t3_with_dates,
        injection_prices  = t3_inj_prices,
        withdrawal_prices = t3_with_prices,
        injection_rate    = 30_000,
        withdrawal_rate   = 35_000,
        max_volume        = 100_000,
        storage_cost_rate = 0.0003,
    )


    # ══════════════════════════════════════════
    # TEST 4 — Capacity constraint is binding
    #          (try to inject more than max_volume allows)
    # ══════════════════════════════════════════
    print("─" * 65)
    print("TEST 4 — Capacity constraint forces partial injection")
    print("─" * 65)

    t4_inj_dates  = [date(2024, 4, 1), date(2024, 4, 15)]  # Two large injections
    t4_with_dates = [date(2024, 10, 1)]

    t4_inj_prices  = [estimate_price(d) for d in t4_inj_dates]
    t4_with_prices = [estimate_price(d) for d in t4_with_dates]

    v4 = price_storage_contract(
        injection_dates   = t4_inj_dates,
        withdrawal_dates  = t4_with_dates,
        injection_prices  = t4_inj_prices,
        withdrawal_prices = t4_with_prices,
        injection_rate    = 80_000,     # tries to inject 80k each time
        withdrawal_rate   = 100_000,
        max_volume        = 100_000,    # but capacity is only 100k total
        storage_cost_rate = 0.0004,
    )

    # ── Summary ──────────────────────────────────────────────────
    print("\n" + "═" * 65)
    print("  SUMMARY OF ALL TESTS")
    print("═" * 65)
    print(f"  Test 1  (seasonal trade)          : ${v1:>10,.2f}")
    print(f"  Test 2  (unprofitable trade)       : ${v2:>10,.2f}")
    print(f"  Test 3  (multi-cycle)              : ${v3:>10,.2f}")
    print(f"  Test 4  (capacity constrained)     : ${v4:>10,.2f}")
    print("═" * 65)


─────────────────────────────────────────────────────────────────
TEST 1 — Basic seasonal trade (inject in summer, withdraw in winter)
─────────────────────────────────────────────────────────────────
Date           Event        Volume Δ      Stock    Cash Flow
-----------------------------------------------------------------
2024-06-01     INJECT        +50,000     50,000 $-421,393.47
2024-07-01     INJECT        +50,000    100,000 $-410,000.00
2024-11-01     WITHDRAW      -50,000     50,000 $+545,082.36
2024-12-01     WITHDRAW      -50,000          0 $+568,539.09

  Withdrawal revenue  : $1,113,621.45
  Injection cost      : $  831,393.47
  Storage cost        : $    7,650.00
  ──────────────────────────────────────
  CONTRACT VALUE      : $  274,577.98

─────────────────────────────────────────────────────────────────
TEST 2 — Unprofitable trade (inject in winter, withdraw in summer)
─────────────────────────────────────────────────────────────────
Date           Event        Volum